# bge-m3 연구실 ↔ 교과목 매칭 실험 v2 (정답 세트 없이 비교)

v1 분석에서 나온 문제 다섯 가지를 고친 버전들을 만들고, **사람이 만든 정답 없이** 비교합니다.

| 버전 | 내용 |
|---|---|
| A | v1 그대로 (비교 기준) |
| B | A + **조각 정리**: 짧은/의미 없는 조각 제거, 강의계획 항목 3개씩 묶기 |
| C | B + **허브 보정**: 모든 연구실과 두루 가까운 과목 감점 |
| D | C + **주제 분산 + 중복 제거**: 한 주제가 목록을 독차지하지 않게, 교차개설 과목 하나로 |
| E | D + **규칙 엔진 가산점**: 기존 개념 일치가 있으면 가산 (하이브리드) |

**정답 없이 비교하는 방법**
1. **자동 건강 지표** (무료): 허브 과목, 주제 커버리지, 짧은 근거, 중복, 타 학과 비율, 은(silver) 정답 재현율
2. **Claude Code 채점** (무료): 버전 정보를 숨긴 채점용 파일을 Claude Code가 0~2점으로 채점
3. **나란히 보기**: 같은 연구실의 A와 최종 버전 목록을 눈으로 비교

### 준비
- **런타임 → 런타임 유형 변경 → T4 GPU**
- 업로드할 파일 4개 (서버 저장소 `data/`): `courses.jsonl`, `lab_profiles.jsonl`, `course_profiles.jsonl`, `concepts.json`

In [ ]:
# 1. GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 선택하고 다시 실행하세요.')

In [ ]:
# 2. 라이브러리 설치
!pip install -q sentence-transformers

In [ ]:
# 3. 데이터 업로드
import os
from google.colab import files
need = ['courses.jsonl', 'lab_profiles.jsonl', 'course_profiles.jsonl', 'concepts.json']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('업로드할 파일:', missing)
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'아직 없는 파일: {missing}'
print('데이터 준비 완료')

In [ ]:
# 4. 모델과 데이터 불러오기
import json, re, time, random, statistics
from collections import defaultdict, Counter
import numpy as np
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device=device)
if device == 'cuda':
    model.half()
model.max_seq_length = 256
print(f'모델 로드 {time.time()-t0:.1f}s, device={device}')

def enc(texts, bs=64):
    return model.encode(texts, batch_size=bs, normalize_embeddings=True, show_progress_bar=len(texts) > 200,
                        convert_to_numpy=True).astype(np.float32)

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

courses = load_jsonl('courses.jsonl')
course_by_id = {c['record_id']: c for c in courses}
profiles = [p for p in load_jsonl('lab_profiles.jsonl') if p.get('status') != 'insufficient']
course_profiles = {c['record_id']: c for c in load_jsonl('course_profiles.jsonl')}
concepts = {c['id']: c for c in json.load(open('concepts.json', encoding='utf-8'))['concepts']}
print(f'과목 {len(courses)}개, 추천 대상 연구실 {len(profiles)}곳')

## 1단계: 과목 조각 만들기 (v1 방식 / 정리한 방식)

- **v1 조각**: 설명, 학습목표, 강의계획 각 줄(15자 초과)
- **정리한 조각**: 30자 미만·의미 없는 줄 제거("See attached file", "Lecture Schedule", "TBD" 등), 강의계획은 3줄씩 묶어 문맥 유지

In [ ]:
def course_name(c):
    return c.get('course_name_kr') or c.get('course_name_en') or c['course_code']

def plan_lines(c):
    return [l.strip() for l in re.split(r'\n+', str(c.get('course_plan') or '')) if l.strip()]

# v1 조각 (버전 A)
def chunks_v1():
    out = []
    for c in courses:
        parts = [('설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]
        parts += [(f'강의계획 {i+1}', p) for i, p in enumerate([p for p in plan_lines(c) if len(p) > 15][:20])]
        for where, text in parts:
            text = text.strip()
            if len(text) <= 15 or re.match(r'^\s*(선수과목|prerequisite)', text, re.I):
                continue
            out.append({'record_id': c['record_id'], 'where': where, 'text': text[:500]})
    return out

JUNK = re.compile(r'(see attached|attached file|첨부|lecture schedule|course schedule|tbd|tba|to be announced|midterm|final exam|중간고사|기말고사|^week\s*\d+\s*:?$|^\d+\s*주차\s*:?$)', re.I)
def meaningful(line):
    body = re.sub(r'^[\W\d_]*(week|주차|lecture|chapter|ch\.)?\s*[\d.]*\s*[:.)-]?\s*', '', line, flags=re.I)
    return len(body) >= 8 and not JUNK.search(line)

# 정리한 조각 (버전 B~E)
MIN_CHARS, PLAN_WINDOW = 30, 3
def chunks_clean():
    out = []
    for c in courses:
        for where, text in [('설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]:
            text = text.strip()
            if len(text) >= MIN_CHARS and not re.match(r'^\s*(선수과목|prerequisite)', text, re.I):
                out.append({'record_id': c['record_id'], 'where': where, 'text': text[:500]})
        lines = [l for l in plan_lines(c) if meaningful(l)]
        for i in range(0, len(lines), PLAN_WINDOW):
            text = ' / '.join(lines[i:i+PLAN_WINDOW])
            if len(text) >= MIN_CHARS:
                out.append({'record_id': c['record_id'], 'where': f'강의계획 {i+1}~{min(i+PLAN_WINDOW, len(lines))}', 'text': text[:500]})
    return out

def encode_chunks(chunks):
    texts = [f"{course_name(course_by_id[m['record_id']])} | {m['text']}" for m in chunks]
    return enc(texts)

meta_A, meta_B = chunks_v1(), chunks_clean()
t0 = time.time()
emb_A, emb_B = encode_chunks(meta_A), encode_chunks(meta_B)
print(f'v1 조각 {len(meta_A)}개, 정리한 조각 {len(meta_B)}개, 벡터화 {time.time()-t0:.1f}s')
print('정리한 조각 예시:')
for m in random.Random(0).sample(meta_B, 3):
    print('  ', course_name(course_by_id[m['record_id']]), '|', m['where'], '|', m['text'][:90])

## 2단계: 연구실 주제 벡터화

In [ ]:
lab_topics = []          # (lab_id, topic)
for p in profiles:
    for t in p['topics']:
        if t.get('weight', 'primary') == 'primary':
            lab_topics.append((p['lab_id'], t))
emb_T = enc([f"{t['label_ko']} | {t['quote']}" for _, t in lab_topics])
topics_of = defaultdict(list)
for row, (lid, _) in enumerate(lab_topics):
    topics_of[lid].append(row)
print(f'연구실 주제 {len(lab_topics)}개')

## 3단계: 버전별 순위 계산

공통 구조: (연구실 주제 × 과목) 점수 → 연구실 × 과목 점수 → 상위 30개 목록과 근거 문장 쌍.
조절값(λ, γ, β)은 아래 셀 맨 위에서 바꿀 수 있습니다.

In [ ]:
HUB_LAMBDA = 0.7     # C: 허브 보정 강도 (과목의 '평균적 가까움'을 얼마나 뺄지)
TOPIC_GAMMA = 0.03   # D: 같은 주제에서 과목을 하나 더 뽑을 때마다 깎는 점수
RULE_BETA = 0.04     # E: 규칙 엔진 개념 일치 가산점 (제목 일치 시 전체, 본문만 일치 시 절반)
GENERIC = {'machine_learning', 'software_engineering', 'statistics_probability', 'optimization'}
TOP_LIST = 30

course_ids = [c['record_id'] for c in courses]
cidx = {rid: i for i, rid in enumerate(course_ids)}

def topic_course_scores(meta, emb, topk=2):
    """(주제 × 과목) 점수 = 그 과목 조각 중 상위 topk개 유사도 평균. 근거 조각 번호도 반환."""
    sim = emb_T @ emb.T                                   # topics x chunks
    owner = np.array([cidx[m['record_id']] for m in meta])
    S = np.full((len(lab_topics), len(courses)), -1.0, dtype=np.float32)
    best_chunk = np.full((len(lab_topics), len(courses)), -1, dtype=np.int64)
    for ci in range(len(courses)):
        cols = np.where(owner == ci)[0]
        if len(cols) == 0:
            continue
        block = sim[:, cols]
        k = min(topk, len(cols))
        part = -np.sort(-block, axis=1)[:, :k]
        S[:, ci] = part.mean(axis=1)
        best_chunk[:, ci] = cols[np.argmax(block, axis=1)]
    return S, best_chunk

def norm_name(name):
    return re.sub(r'[\s\-·]+', '', name).lower()

def rule_bonus(lab_id, rid):
    prof = next(p for p in profiles if p['lab_id'] == lab_id)
    core = {t['concept'] for t in prof['topics'] if t.get('weight', 'primary') == 'primary' and t.get('scope', 'core') == 'core'}
    shared = [x for x in course_profiles.get(rid, {}).get('concepts', []) if x['concept'] in core]
    if not shared:
        return 0.0, []
    specific = [x for x in shared if x['concept'] not in GENERIC]
    if not specific:
        return RULE_BETA * 0.25, shared
    return RULE_BETA * (1.0 if any(x['strength'] == 'title' for x in specific) else 0.5), shared

def build_entry(lab_id, ci, score, row, chunk_meta, chunk_i):
    c = courses[ci]; t = lab_topics[row][1]; m = chunk_meta[chunk_i]
    return {'record_id': c['record_id'], 'code': c['course_code'], 'name': course_name(c), 'score': round(float(score), 4),
            'lab_topic': t['label_ko'], 'lab_quote': t['quote'][:160], 'course_where': m['where'], 'course_text': m['text'][:160]}

# ---- 버전 A: v1 방식 그대로 (조각 쌍 상위 3개 평균 + 여러 주제 가산) ----
def rank_v1():
    sim = emb_T @ emb_A.T
    out = {}
    for lid, rows in topics_of.items():
        best = defaultdict(list)
        for row in rows:
            for k in np.argsort(-sim[row])[:80]:
                best[cidx[meta_A[k]['record_id']]].append((float(sim[row, k]), row, int(k)))
        scored = []
        for ci, hits in best.items():
            hits.sort(reverse=True)
            n_topics = len({h[1] for h in hits if h[0] > 0.5})
            s = sum(h[0] for h in hits[:3]) / 3 + 0.03 * max(0, n_topics - 1)
            scored.append(build_entry(lid, ci, s, hits[0][1], meta_A, hits[0][2]))
        out[lid] = sorted(scored, key=lambda x: -x['score'])[:TOP_LIST]
    return out

S_B, chunk_B = topic_course_scores(meta_B, emb_B)
background = S_B.mean(axis=0)            # 과목이 모든 연구실 주제와 평균적으로 얼마나 가까운지

def rank_variant(hub=False, diversify=False, dedupe=False, rule=False):
    S = S_B - HUB_LAMBDA * background[None, :] if hub else S_B
    out = {}
    for lid, rows in topics_of.items():
        sub = S[rows]                                       # lab topics x courses
        best_row = sub.argmax(axis=0)
        cand = []
        for ci in range(len(courses)):
            r = rows[best_row[ci]]
            s = float(sub[best_row[ci], ci])
            if rule:
                b, _ = rule_bonus(lid, courses[ci]['record_id']); s += b
            cand.append([s, ci, r])
        cand.sort(key=lambda x: -x[0])
        cand = cand[:200]
        picked, used_topic, seen_names = [], Counter(), set()
        pool = cand[:]
        while pool and len(picked) < TOP_LIST:
            if diversify:
                j = max(range(len(pool)), key=lambda i: pool[i][0] - TOPIC_GAMMA * used_topic[pool[i][2]])
            else:
                j = 0
            s, ci, r = pool.pop(j)
            if dedupe:
                key = norm_name(course_name(courses[ci]))
                if key in seen_names:
                    continue
                seen_names.add(key)
            used_topic[r] += 1
            picked.append(build_entry(lid, ci, s, r, meta_B, chunk_B[r, ci]))
        out[lid] = picked
    return out

t0 = time.time()
variants = {
    'A': rank_v1(),
    'B': rank_variant(),
    'C': rank_variant(hub=True),
    'D': rank_variant(hub=True, diversify=True, dedupe=True),
    'E': rank_variant(hub=True, diversify=True, dedupe=True, rule=True),
}
print(f'버전 {list(variants)} 계산 {time.time()-t0:.1f}s')

## 4단계: 자동 건강 지표 (정답 없이)

| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| 은정답 재현율 | 규칙 엔진에서 **과목 제목**에 연구실 핵심 주제가 들어간 과목이 상위 10개에 들어간 비율 (사람이 봐도 거의 확실한 연결) | ↑ (단, E는 규칙을 섞어서 유리함) |
| 최대 허브 | 한 과목이 상위 10개에 등장한 연구실 비율의 최댓값 | ↓ |
| 다양성 | 전체 연구실 상위 10개에 등장한 서로 다른 과목 수 | ↑ |
| 주제 커버리지 | 상위 10개가 다룬 연구실 주제 수 ÷ min(주제 수, 10) | ↑ |
| 한 주제 독점 | 상위 10개 중 7개 이상이 한 주제인 연구실 수 | ↓ |
| 짧은 근거 | 근거 과목 조각이 35자 미만인 비율 | ↓ |
| 중복 | 같은 이름 과목이 상위 10개에 두 번 이상 | ↓ |
| 4개 학과·수학 외 | CSED·EECE·CITE·SEMI·DISU·MATH 외 과목 비율 (참고용, 맞는 경우도 있음) | 참고 |

In [ ]:
TOPN = 10
CORE_PREFIX = {'CSED', 'EECE', 'CITE', 'SEMI', 'DISU', 'MATH'}

silver = {}
for p in profiles:
    lid = p['lab_id']
    core = {t['concept'] for t in p['topics'] if t.get('weight', 'primary') == 'primary' and t.get('scope', 'core') == 'core'} - GENERIC
    s = {rid for rid, cp in course_profiles.items() if any(x['concept'] in core and x['strength'] == 'title' for x in cp['concepts'])}
    if s:
        silver[lid] = s

def metrics(ranking):
    labs = list(ranking)
    tops = {lid: ranking[lid][:TOPN] for lid in labs}
    appear = Counter(c['record_id'] for lst in tops.values() for c in lst)
    rec = [len(silver[l] & {c['record_id'] for c in tops[l]}) / min(TOPN, len(silver[l])) for l in labs if l in silver]
    cov, hog = [], 0
    for lid in labs:
        n_topics = len(topics_of[lid])
        tc = Counter(c['lab_topic'] for c in tops[lid])
        cov.append(len(tc) / min(n_topics, TOPN))
        hog += tc.most_common(1)[0][1] >= 7 if tc else 0
    allc = [c for lst in tops.values() for c in lst]
    dups = sum(len(lst) - len({norm_name(c['name']) for c in lst}) for lst in tops.values())
    return {
        '은정답 재현율': round(statistics.mean(rec), 3),
        '최대 허브': f"{max(appear.values())}/{len(labs)}",
        '다양성(서로 다른 과목)': len(appear),
        '주제 커버리지': round(statistics.mean(cov), 3),
        '한 주제 독점(곳)': hog,
        '짧은 근거': round(sum(len(c['course_text']) < 35 for c in allc) / len(allc), 3),
        '중복': dups,
        '4개 학과·수학 외': round(sum(c['code'][:4] not in CORE_PREFIX for c in allc) / len(allc), 3),
    }

import pandas as pd
table = pd.DataFrame({v: metrics(r) for v, r in variants.items()})
print(f'은정답이 있는 연구실 {len(silver)}곳 / 전체 {len(profiles)}곳')
table

In [ ]:
# 버전별로 가장 많이 반복 등장한 과목 (허브 확인)
for v, r in variants.items():
    appear = Counter((c['code'], c['name']) for lst in r.values() for c in lst[:TOPN])
    print(f"[{v}] " + ', '.join(f"{n}곳 {k[1]}" for k, n in appear.most_common(5)))

In [ ]:
# 버전 간 상위 10개 겹침 (Jaccard 평균) — 어떤 수정이 결과를 크게 바꿨는지
names = list(variants)
jac = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        vals = []
        for lid in variants[a]:
            x = {c['record_id'] for c in variants[a][lid][:TOPN]}; y = {c['record_id'] for c in variants[b][lid][:TOPN]}
            vals.append(len(x & y) / len(x | y))
        jac.loc[a, b] = round(statistics.mean(vals), 2)
jac

## 5단계: Claude Code 채점용 파일 만들기 (블라인드)

- 무작위 연구실 `N_JUDGE_LABS`곳에 대해 **모든 버전의 상위 10개 과목을 합친 뒤 섞어서** 채점용 파일(`judge_packet.json`)을 만듭니다.
- 채점용 파일에는 과목이 **어느 버전에서 나왔는지 들어 있지 않습니다**. 버전 대응표는 `judge_key.json`에 따로 저장합니다.
- 이 zip을 Claude Code에 주면 `judge_packet.json`만 보고 과목마다 0~2점을 매겨 `judge_scores.json`을 돌려줍니다.
  - **2** 연구실 주제에 직접 필요한 지식·방법
  - **1** 기초 지식이거나 부분적으로 관련
  - **0** 무관하거나 같은 단어를 다른 뜻으로 쓴 경우
- 받은 `judge_scores.json`을 7단계에서 업로드하면 버전별 점수가 계산됩니다.
- AI 채점도 정답은 아닙니다. 점수가 엇갈리는 몇 개는 눈으로 확인해 보세요.

In [ ]:
N_JUDGE_LABS = 20
SEED = 7
prof_by_id = {p['lab_id']: p for p in profiles}
judge_labs = random.Random(SEED).sample([p['lab_id'] for p in profiles], min(N_JUDGE_LABS, len(profiles)))

packet, key = [], {}
for lid in judge_labs:
    p = prof_by_id[lid]
    rids = list(dict.fromkeys(c['record_id'] for r in variants.values() for c in r[lid][:TOPN]))
    random.Random(lid).shuffle(rids)
    items = []
    for i, rid in enumerate(rids):
        c = course_by_id[rid]
        items.append({'id': f'{lid}#C{i:02d}', 'code': c['course_code'], 'name_kr': c.get('course_name_kr') or '',
                      'name_en': c.get('course_name_en') or '',
                      'description': (c.get('description') or '')[:400], 'objectives': (c.get('objectives') or '')[:300]})
        key[f'{lid}#C{i:02d}'] = rid
    packet.append({'lab_id': lid, 'professors': p['professors'], 'summary_ko': p['summary_ko'],
                   'topics': [{'label': t['label_ko'], 'quote': t['quote'][:250]} for t in p['topics'] if t.get('weight', 'primary') == 'primary'],
                   'courses': items})
json.dump({'instructions': '각 course에 0~2점: 2=연구실 주제에 직접 필요한 지식·방법, 1=기초 또는 부분 관련, 0=무관/단어만 겹침. '
                           '반환 형식: {"<course id>": {"score": 0|1|2, "reason": "한 문장"}}',
           'labs': packet}, open('judge_packet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(key, open('judge_key.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'채점용 파일: 연구실 {len(packet)}곳, 과목-연구실 쌍 {len(key)}개 → judge_packet.json')

## 7단계 (채점 후): Claude Code 채점 결과 불러오기

Claude Code가 돌려준 `judge_scores.json`을 업로드하면 버전별 점수를 계산합니다. 아직 없으면 건너뛰세요.

In [ ]:
judged = {}   # (lab_id, record_id) -> (score, reason)
if not os.path.exists('judge_scores.json'):
    print('judge_scores.json 이 없으면 업로드 창에서 취소하고 넘어가도 됩니다.')
    try:
        files.upload()
    except Exception:
        pass
if os.path.exists('judge_scores.json'):
    key = json.load(open('judge_key.json', encoding='utf-8'))
    scores = json.load(open('judge_scores.json', encoding='utf-8'))
    for cid, v in scores.items():
        if cid in key:
            judged[(cid.split('#')[0], key[cid])] = (int(v['score']), v.get('reason', ''))
    print(f'채점된 쌍 {len(judged)} / {len(key)}')

if judged:
    rows = {}
    for v, r in variants.items():
        sc = [judged[(lid, c['record_id'])][0] for lid in judge_labs for c in r[lid][:TOPN] if (lid, c['record_id']) in judged]
        rows[v] = {'평균 점수(0~2)': round(statistics.mean(sc), 3),
                   '2점 비율': round(sum(s == 2 for s in sc) / len(sc), 3),
                   '0점 비율': round(sum(s == 0 for s in sc) / len(sc), 3),
                   '채점된 쌍': len(sc)}
    display(pd.DataFrame(rows))
    print('\n0점을 받은 과목 예시 (최종 버전 E):')
    shown = 0
    for lid in judge_labs:
        for c in variants['E'][lid][:TOPN]:
            s = judged.get((lid, c['record_id']))
            if s and s[0] == 0 and shown < 10:
                print(f"  {prof_by_id[lid]['professors'][0]} ← {c['code']} {c['name']}: {s[1]}"); shown += 1
else:
    print('채점 결과 없음 — 6단계까지의 지표만 확인하세요.')

## 8단계: 나란히 보기 (A vs 최종 E)

In [ ]:
def side_by_side(prof, n=10):
    p = next((x for x in profiles if any(prof in name for name in x['professors'])), None)
    if p is None:
        print(f'{prof}: 결과 없음'); return
    lid = p['lab_id']
    print(f"\n■ {' · '.join(p['professors'])} — {p['summary_ko']}")
    a, e = variants['A'][lid][:n], variants['E'][lid][:n]
    for i in range(n):
        left = f"{a[i]['code']} {a[i]['name']}"[:34] if i < len(a) else ''
        right = f"{e[i]['code']} {e[i]['name']}" if i < len(e) else ''
        mark = ''
        if i < len(e) and (lid, e[i]['record_id']) in judged:
            mark = f"  (AI {judged[(lid, e[i]['record_id'])][0]}점)"
        print(f"  {i+1:>2}. A: {left:<36} | E: {right}{mark}")
    print('  E의 근거 예시:')
    for c in e[:3]:
        print(f"     {c['name']}: [{c['lab_topic']}] ↔ ({c['course_where']}) {c['course_text'][:70]}")

for prof in ['한수희', '김광인', '서영주', '이근배']:
    side_by_side(prof)

## 결과 저장·다운로드

In [ ]:
import zipfile
summary = {v: {lid: r[lid] for lid in r} for v, r in variants.items()}
json.dump(summary, open('rankings_v2.json', 'w', encoding='utf-8'), ensure_ascii=False)
table.to_csv('metrics_v2.csv', encoding='utf-8-sig')
json.dump({'HUB_LAMBDA': HUB_LAMBDA, 'TOPIC_GAMMA': TOPIC_GAMMA, 'RULE_BETA': RULE_BETA, 'MIN_CHARS': MIN_CHARS,
           'PLAN_WINDOW': PLAN_WINDOW, 'N_JUDGE_LABS': N_JUDGE_LABS, 'SEED': SEED},
          open('settings_v2.json', 'w'), indent=1)
with zipfile.ZipFile('bge_m3_results_v2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['rankings_v2.json', 'metrics_v2.csv', 'settings_v2.json', 'judge_packet.json', 'judge_key.json']:
        z.write(f)
print('저장 완료: bge_m3_results_v2.zip')
files.download('bge_m3_results_v2.zip')